# 2: A new star

Here I treat V624 Tau as a new star and go through every step you need for your own star: the mode table, the star file, the priors, the covariance matrix and the fit. The covariance matrix shipped for V624 Tau is used only as a check, in step 5.

In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt

from velociscuti import load_emulator, velociscuti_sampler, build_covariance
from velociscuti import uniform_prior, gaussian_prior, posterior_plot, posterior_summary
from velociscuti.utils import load_modes

## 1. The mode table

`stars/<name>/<name>_modes.csv` has one row per extracted frequency. The sampler reads six columns and ignores any others:

| column | meaning |
|---|---|
| `label` | any name for the peak |
| `f_obs` | frequency in cycles per day |
| `freq_err` | uncertainty of the frequency in cycles per day |
| `n_obs` | radial order: positive for p modes, 0 for f modes, negative for g modes |
| `l_obs` | degree $\ell$. A negative value marks the peak as unidentified |
| `m_obs` | azimuthal order, positive for prograde modes |

A row is used only when `l_obs >= 0`. Unidentified peaks can stay in the table with `l_obs = -1`, and the sampler skips them. `n_obs` cannot serve as the flag, because `n_obs = -1` is also the radial order of a real g mode.

The emulator predicts p modes with `n_obs` from 1 to 11 and $\ell$ from 0 to 3, f modes with $\ell$ = 2 and 3, and g modes with `n_obs` = $-1$ and $-2$ and $\ell$ from 1 to 3, each for every $m$ from $-\ell$ to $\ell$. A mode outside this set raises an error, and so do two frequencies assigned to the same mode.

`stars/TEMPLATE/` holds a small example with one row of each kind (the numbers are placeholders):

In [ ]:
with open('stars/TEMPLATE/TEMPLATE_modes.csv', 'r') as fp:
    print(fp.read())

`load_modes` returns the modes the sampler will use. For V624 Tau it keeps 26 of the 117 rows in the table:

In [ ]:
mode_cols, freqs, freq_errs = load_modes('V624_Tau')
for col, freq, freq_err in zip(mode_cols, freqs, freq_errs):
    print(f'{col:12s} {freq:9.5f} +/- {freq_err:.5f}')

## 2. The star file

`stars/<name>/<name>.json` holds a display name and the effective temperature in kelvin as `[value, uncertainty]`:

In [ ]:
with open('stars/V624_Tau/V624_Tau.json', 'r') as fp:
    print(fp.read())

## 3. The priors

The priors come before the covariance, because the covariance describes the emulator's error over the region of parameter space that the priors allow. Build the covariance and run the sampler with the same list of priors, in the order `m`, `z`, `Myr`, `v_eq`.

Any frozen `scipy.stats` distribution works as a prior. `uniform_prior` and `gaussian_prior` are shortcuts for the two kinds used in the paper. The limits have to stay inside the range the emulator was trained on, and the sampler checks that when it starts. Below are the priors of the paper, written out.

In [ ]:
priors = [
    uniform_prior(1.4, 2.5),                                                      ## mass [Msun]
    gaussian_prior(0.0152, 0.00189, 0.001, 0.026),     ## Z: mean, sd, lower and upper limit
    uniform_prior(50, 200),                                                       ## age [Myr]
    uniform_prior(0, 200),                                                        ## v_eq [km/s]
]

## 4. The covariance matrix

The likelihood needs $C_\nu = C_{\rm obs} + C_{\rm emu}$. $C_{\rm obs}$ is diagonal and holds `freq_err` squared. $C_{\rm emu}$ is the covariance of the emulator's frequency errors: how far, and how coherently across modes, the emulator's frequencies miss those of the stellar models it was trained to reproduce.

I estimate $C_{\rm emu}$ from grid models that were held out of training. `build_covariance` takes the held-out residuals (grid model minus emulator) of the identified modes, keeps the rows inside the prior limits, weights them by the prior density, shrinks the covariance 10 per cent of the way towards its diagonal, and rescales it so that the diagonal holds the squared emulator errors stored in `sigma_model.npy`, keeping the correlations. $T_{\rm eff}$ gets its own term, built from its observational uncertainty and the emulator error in the same way.

The repository ships the held-out residuals inside the prior box of the paper (`calibration/pleiades_box.npz`), which is all that the priors above need.

In [ ]:
built = build_covariance('V624_Tau', priors=priors, save_as='V624_Tau_covariance_rebuilt.npz')

## 5. A check against the shipped matrix

V624 Tau is not really new, so I can compare the matrix I just built with the one used in the paper. The estimator and the held-out models are the same, so the two should agree to within rounding.

In [ ]:
def correlation(C):
    sd = np.sqrt(np.diag(C))
    return C / np.outer(sd, sd)

with np.load('stars/V624_Tau/V624_Tau_covariance.npz') as shipped:
    C_emu_paper = shipped['C_emu']

print('largest difference in correlation:          ', np.max(np.abs(correlation(built['C_emu']) - correlation(C_emu_paper))))
print('largest relative difference on the diagonal:', np.max(np.abs(np.diag(built['C_emu']) / np.diag(C_emu_paper) - 1)))

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation(built['C_emu']), vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(built['mode_cols'])))
ax.set_xticklabels(built['mode_cols'], rotation=90, fontsize=6)
ax.set_yticks(range(len(built['mode_cols'])))
ax.set_yticklabels(built['mode_cols'], fontsize=6)
fig.colorbar(image, label='correlation of the emulator error')
plt.show()

## 6. The fit

The sampler takes the same priors, and `covariance=` points it to the file I just made. Without that argument it looks for `<name>_covariance.npz`, which is the name `build_covariance` uses by default.

In [ ]:
run_settings = dict()   ## for a quick look: dict(min_num_live_points=200, dlogz=0.5, min_ess=200, frac_remain=0.1)

In [ ]:
emulator = load_emulator()
sampler = velociscuti_sampler(emulator, priors=priors)

results = sampler('V624_Tau', covariance='V624_Tau_covariance_rebuilt.npz', save_as='V624_Tau_newstar_results.npz', **run_settings)

In [ ]:
posterior_plot(results, star_name='V624 Tau');

In [ ]:
age = posterior_summary(results)['Myr']
print(f"age here:  {age['median']:.1f} +{age['p84'] - age['median']:.1f} -{age['median'] - age['p16']:.1f} Myr")
print('age paper: 116.9 +7.7 -8.2 Myr')

## 7. Your own star

1. Copy `stars/TEMPLATE` to `stars/MyStar`, rename the two files to `MyStar.json` and `MyStar_modes.csv`, and fill them in.
2. Choose the priors. The mass and $Z$ limits of the paper already span the whole grid. If your age prior reaches outside 50 to 200 Myr, or your velocity prior above 200 km/s, `build_covariance` needs the full set of held-out residuals. That set is about 2 GB and is kept in Git LFS. Fetch it once with `git lfs pull --include="calibration/full" --exclude=""`.
3. Run `build_covariance('MyStar', priors=priors)` and then `sampler('MyStar')`, with the sampler initialised with the same priors.

Three things to keep in mind. The priors have to stay inside the range the emulator was trained on, which is listed, with a note on the ages the grid covers, in `velociscuti/model/velociscuti_info.md`. Tight priors leave few held-out models to estimate the covariance from, and `build_covariance` stops when there are too few. And the fit takes the mode identifications as given and does not test them.